# Provider-agnostic LLM client — Colab edition

One interface (`chat()` / `stream()`) over **hosted APIs (Gemini, Anthropic)** and a **local Ollama model running inside this Colab VM**.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (Ollama works on CPU too, just much slower).

Run the cells top to bottom.

## 1. Write the client module

In [8]:
%%writefile llm_client.py
"""
llm_client.py — one interface over hosted LLM APIs (Anthropic, Gemini) and a local Ollama model.

    from llm_client import create_client, Message

    llm = create_client("ollama:llama3.2")                 # local
    llm = create_client("anthropic:claude-sonnet-5-5")     # hosted (needs ANTHROPIC_API_KEY)
    llm = create_client("gemini:gemini-2.5-flash-lite")    # hosted (needs GEMINI_API_KEY)

    resp = llm.chat([Message.user("Say hi")], system="Be brief.", max_tokens=100)
    print(resp.text, resp.usage)

    for chunk in llm.stream([Message.user("Count to 5")]):
        print(chunk, end="", flush=True)

Calling code depends only on the `LLMClient` protocol, so swapping providers is a
one-string change. Standard library only (no requests/httpx/SDKs).
"""
from __future__ import annotations

import json
import os
import random
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from typing import Any, Iterator, Literal, Protocol, Sequence, runtime_checkable

__all__ = [
    "Message", "Usage", "ChatResponse", "LLMError", "LLMClient",
    "AnthropicClient", "GeminiClient", "OllamaClient", "FallbackClient", "create_client",
]

Role = Literal["system", "user", "assistant"]
FinishReason = Literal["stop", "length", "other"]


# --------------------------------------------------------------------------- #
# Common types
# --------------------------------------------------------------------------- #
@dataclass(frozen=True)
class Message:
    role: Role
    content: str

    @classmethod
    def system(cls, content: str) -> "Message":
        return cls("system", content)

    @classmethod
    def user(cls, content: str) -> "Message":
        return cls("user", content)

    @classmethod
    def assistant(cls, content: str) -> "Message":
        return cls("assistant", content)


@dataclass
class Usage:
    input_tokens: int | None = None
    output_tokens: int | None = None


@dataclass
class ChatResponse:
    text: str
    model: str
    provider: str
    finish_reason: FinishReason
    usage: Usage = field(default_factory=Usage)
    raw: dict[str, Any] = field(default_factory=dict, repr=False)


class LLMError(Exception):
    """Every provider failure surfaces as this one exception type."""

    def __init__(self, message: str, *, provider: str, status: int | None = None,
                 retryable: bool = False):
        super().__init__(message)
        self.provider = provider
        self.status = status
        self.retryable = retryable


@runtime_checkable
class LLMClient(Protocol):
    provider: str
    model: str

    def chat(self, messages: Sequence[Message], *, system: str | None = None,
             max_tokens: int = 1024, temperature: float | None = None,
             stop: Sequence[str] | None = None) -> ChatResponse: ...

    def stream(self, messages: Sequence[Message], *, system: str | None = None,
               max_tokens: int = 1024, temperature: float | None = None,
               stop: Sequence[str] | None = None) -> Iterator[str]: ...


# --------------------------------------------------------------------------- #
# Shared HTTP plumbing
# --------------------------------------------------------------------------- #
class _HTTPClient:
    provider = "base"

    def __init__(self, model: str, base_url: str, *, timeout: float = 120.0,
                 max_retries: int = 2):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries

    def _headers(self) -> dict[str, str]:
        return {"content-type": "application/json"}

    @staticmethod
    def _backoff(attempt: int, retry_after: str | None) -> float:
        if retry_after:
            try:
                return min(float(retry_after), 30.0)
            except ValueError:
                pass
        return min(0.5 * 2 ** attempt, 8.0) + random.uniform(0, 0.25)

    def _post(self, path: str, body: dict[str, Any]):
        """POST JSON, retrying 408/409/429/5xx and connection errors. Returns the open response."""
        url = self.base_url + path
        data = json.dumps(body).encode()
        for attempt in range(self.max_retries + 1):
            req = urllib.request.Request(url, data=data, headers=self._headers(), method="POST")
            try:
                return urllib.request.urlopen(req, timeout=self.timeout)
            except urllib.error.HTTPError as e:
                detail = e.read().decode(errors="replace")
                retryable = e.code in (408, 409, 429) or e.code >= 500
                if retryable and attempt < self.max_retries:
                    time.sleep(self._backoff(attempt, e.headers.get("retry-after")))
                    continue
                raise LLMError(f"{self.provider} HTTP {e.code}: {detail[:500]}",
                               provider=self.provider, status=e.code,
                               retryable=retryable) from e
            except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
                if attempt < self.max_retries:
                    time.sleep(self._backoff(attempt, None))
                    continue
                reason = getattr(e, "reason", e)
                raise LLMError(f"{self.provider}: cannot reach {url} ({reason})",
                               provider=self.provider, retryable=True) from e
        raise AssertionError("unreachable")

    def _post_json(self, path: str, body: dict[str, Any]) -> dict[str, Any]:
        with self._post(path, body) as resp:
            return json.loads(resp.read())

    @staticmethod
    def _split_system(messages: Sequence[Message], system: str | None
                      ) -> tuple[str | None, list[Message]]:
        """Merge the `system=` kwarg and any system-role messages into one string."""
        parts = [system] if system else []
        rest = []
        for m in messages:
            (parts.append(m.content) if m.role == "system" else rest.append(m))
        return ("\n\n".join(parts) or None), rest


# --------------------------------------------------------------------------- #
# Hosted: Anthropic Messages API
# --------------------------------------------------------------------------- #
class AnthropicClient(_HTTPClient):
    provider = "anthropic"
    _FINISH = {"end_turn": "stop", "stop_sequence": "stop", "max_tokens": "length"}

    def __init__(self, model: str = "claude-sonnet-5-5", *, api_key: str | None = None,
                 base_url: str = "https://api.anthropic.com", api_version: str = "2023-06-01",
                 **kw):
        super().__init__(model, base_url, **kw)
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        if not self.api_key:
            raise LLMError("ANTHROPIC_API_KEY is not set", provider=self.provider)
        self.api_version = api_version

    def _headers(self) -> dict[str, str]:
        return {"content-type": "application/json", "x-api-key": self.api_key,
                "anthropic-version": self.api_version}

    def _body(self, messages, system, max_tokens, temperature, stop, stream):
        sys_text, msgs = self._split_system(messages, system)
        body: dict[str, Any] = {
            "model": self.model, "max_tokens": max_tokens, "stream": stream,
            "messages": [{"role": m.role, "content": m.content} for m in msgs],
        }
        if sys_text:
            body["system"] = sys_text
        if temperature is not None:
            body["temperature"] = temperature
        if stop:
            body["stop_sequences"] = list(stop)
        return body

    def chat(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        raw = self._post_json("/v1/messages",
                              self._body(messages, system, max_tokens, temperature, stop, False))
        text = "".join(b.get("text", "") for b in raw.get("content", []) if b.get("type") == "text")
        u = raw.get("usage", {})
        return ChatResponse(
            text=text, model=raw.get("model", self.model), provider=self.provider,
            finish_reason=self._FINISH.get(raw.get("stop_reason"), "other"),
            usage=Usage(u.get("input_tokens"), u.get("output_tokens")), raw=raw)

    def stream(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        body = self._body(messages, system, max_tokens, temperature, stop, True)
        with self._post("/v1/messages", body) as resp:
            for line in resp:
                line = line.decode().strip()
                if not line.startswith("data:"):
                    continue
                event = json.loads(line[5:].strip())
                etype = event.get("type")
                if etype == "content_block_delta" and event["delta"].get("type") == "text_delta":
                    yield event["delta"]["text"]
                elif etype == "error":
                    err = event.get("error", {})
                    raise LLMError(f"anthropic stream error: {err.get('message', err)}",
                                   provider=self.provider,
                                   retryable=err.get("type") == "overloaded_error")
                elif etype == "message_stop":
                    return


# --------------------------------------------------------------------------- #
# Local: Ollama
# --------------------------------------------------------------------------- #
class OllamaClient(_HTTPClient):
    provider = "ollama"

    def __init__(self, model: str = "llama3.2", *, base_url: str | None = None,
                 keep_alive: str | None = None, **kw):
        kw.setdefault("max_retries", 1)  # a local server is either up or it isn't
        super().__init__(model, base_url or os.environ.get("OLLAMA_HOST", "http://localhost:11434"), **kw)
        self.keep_alive = keep_alive

    def _body(self, messages, system, max_tokens, temperature, stop, stream):
        sys_text, msgs = self._split_system(messages, system)
        wire = ([{"role": "system", "content": sys_text}] if sys_text else []) + \
               [{"role": m.role, "content": m.content} for m in msgs]
        options: dict[str, Any] = {"num_predict": max_tokens}
        if temperature is not None:
            options["temperature"] = temperature
        if stop:
            options["stop"] = list(stop)
        body: dict[str, Any] = {"model": self.model, "messages": wire,
                                "stream": stream, "options": options}
        if self.keep_alive is not None:
            body["keep_alive"] = self.keep_alive
        return body

    @staticmethod
    def _finish(raw: dict[str, Any]) -> FinishReason:
        return {"stop": "stop", "length": "length"}.get(raw.get("done_reason", "stop"), "other")

    def _check(self, raw: dict[str, Any]) -> None:
        if "error" in raw:
            raise LLMError(f"ollama: {raw['error']}", provider=self.provider)

    def chat(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        raw = self._post_json("/api/chat",
                              self._body(messages, system, max_tokens, temperature, stop, False))
        self._check(raw)
        return ChatResponse(
            text=raw.get("message", {}).get("content", ""), model=raw.get("model", self.model),
            provider=self.provider, finish_reason=self._finish(raw),
            usage=Usage(raw.get("prompt_eval_count"), raw.get("eval_count")), raw=raw)

    def stream(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        body = self._body(messages, system, max_tokens, temperature, stop, True)
        with self._post("/api/chat", body) as resp:
            for line in resp:  # NDJSON: one object per line
                if not line.strip():
                    continue
                chunk = json.loads(line)
                self._check(chunk)
                text = chunk.get("message", {}).get("content", "")
                if text:
                    yield text
                if chunk.get("done"):
                    return


# --------------------------------------------------------------------------- #
# Hosted: Google Gemini API
# --------------------------------------------------------------------------- #
class GeminiClient(_HTTPClient):
    provider = "gemini"
    _FINISH = {"STOP": "stop", "MAX_TOKENS": "length"}

    def __init__(self, model: str = "gemini-2.5-flash-lite", *, api_key: str | None = None,
                 base_url: str = "https://generativelanguage.googleapis.com", **kw):
        super().__init__(model, base_url, **kw)
        self.api_key = api_key or os.environ.get("GEMINI_API_KEY")
        if not self.api_key:
            raise LLMError("GEMINI_API_KEY is not set", provider=self.provider)

    def _headers(self) -> dict[str, str]:
        return {"content-type": "application/json", "x-goog-api-key": self.api_key}

    def _body(self, messages, system, max_tokens, temperature, stop):
        sys_text, msgs = self._split_system(messages, system)
        config: dict[str, Any] = {"maxOutputTokens": max_tokens}
        if temperature is not None:
            config["temperature"] = temperature
        if stop:
            config["stopSequences"] = list(stop)
        body: dict[str, Any] = {
            "contents": [{"role": "model" if m.role == "assistant" else "user",
                          "parts": [{"text": m.content}]} for m in msgs],
            "generationConfig": config,
        }
        if sys_text:
            body["systemInstruction"] = {"parts": [{"text": sys_text}]}
        return body

    @staticmethod
    def _text(raw: dict[str, Any]) -> str:
        cands = raw.get("candidates") or [{}]
        return "".join(p.get("text", "") for p in cands[0].get("content", {}).get("parts", []))

    def chat(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        raw = self._post_json(f"/v1beta/models/{self.model}:generateContent",
                              self._body(messages, system, max_tokens, temperature, stop))
        u = raw.get("usageMetadata", {})
        reason = (raw.get("candidates") or [{}])[0].get("finishReason")
        return ChatResponse(
            text=self._text(raw), model=raw.get("modelVersion", self.model), provider=self.provider,
            finish_reason=self._FINISH.get(reason, "other"),
            usage=Usage(u.get("promptTokenCount"), u.get("candidatesTokenCount")), raw=raw)

    def stream(self, messages, *, system=None, max_tokens=1024, temperature=None, stop=None):
        body = self._body(messages, system, max_tokens, temperature, stop)
        with self._post(f"/v1beta/models/{self.model}:streamGenerateContent?alt=sse", body) as resp:
            for line in resp:
                line = line.decode().strip()
                if line.startswith("data:"):
                    text = self._text(json.loads(line[5:].strip()))
                    if text:
                        yield text


# --------------------------------------------------------------------------- #
# Composition: try one provider, fall back to the next
# --------------------------------------------------------------------------- #
class FallbackClient:
    """Tries each client in order; moves on only for retryable failures (outage, rate limit,
    unreachable server). Bad requests and auth errors are raised immediately."""

    provider = "fallback"

    def __init__(self, *clients: LLMClient):
        if not clients:
            raise ValueError("FallbackClient needs at least one client")
        self.clients = clients
        self.model = clients[0].model

    def chat(self, messages, **kw) -> ChatResponse:
        last: LLMError | None = None
        for c in self.clients:
            try:
                return c.chat(messages, **kw)
            except LLMError as e:
                if not e.retryable:
                    raise
                last = e
        raise last  # type: ignore[misc]

    def stream(self, messages, **kw) -> Iterator[str]:
        last: LLMError | None = None
        for c in self.clients:
            started = False
            try:
                for chunk in c.stream(messages, **kw):
                    started = True
                    yield chunk
                return
            except LLMError as e:
                if started or not e.retryable:  # never splice two models' output together
                    raise
                last = e
        raise last  # type: ignore[misc]


# --------------------------------------------------------------------------- #
# Factory
# --------------------------------------------------------------------------- #
_PROVIDERS: dict[str, type[_HTTPClient]] = {
    "anthropic": AnthropicClient, "gemini": GeminiClient, "ollama": OllamaClient}


def create_client(spec: str, **kwargs) -> LLMClient:
    """Build a client from "provider:model", e.g. "ollama:llama3.2:3b" or
    "anthropic:claude-sonnet-5-5". The model part is optional ("ollama")."""
    provider, _, model = spec.partition(":")
    cls = _PROVIDERS.get(provider.lower())
    if cls is None:
        raise ValueError(f"unknown provider {provider!r}; expected one of {sorted(_PROVIDERS)}")
    return cls(model, **kwargs) if model else cls(**kwargs)

Writing llm_client.py


## 2. (Optional) Run the offline tests
Uses a fake server, so no API key or Ollama needed.

In [9]:
%%writefile test_llm_client.py
"""Tests against an in-process fake server that speaks both the Anthropic and Ollama wire
formats. Run: python -m unittest -v test_llm_client"""
import json
import threading
import unittest
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from llm_client import (AnthropicClient, FallbackClient, GeminiClient, LLMClient, LLMError, Message,
                        OllamaClient, create_client)


class FakeHandler(BaseHTTPRequestHandler):
    fail_next: list[int] = []      # status codes to return before succeeding
    last_body: dict = {}
    last_headers: dict = {}

    def log_message(self, *a):
        pass

    def _send(self, status, payload, ctype="application/json"):
        data = payload if isinstance(payload, bytes) else json.dumps(payload).encode()
        self.send_response(status)
        self.send_header("content-type", ctype)
        self.send_header("content-length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["content-length"])))
        FakeHandler.last_body, FakeHandler.last_headers = body, self.headers
        if FakeHandler.fail_next:
            return self._send(FakeHandler.fail_next.pop(0), {"error": "boom"})
        if self.path == "/v1/messages":
            if body["stream"]:
                events = [
                    {"type": "message_start", "message": {"usage": {"input_tokens": 5}}},
                    {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}},
                    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Hel"}},
                    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "lo"}},
                    {"type": "message_stop"},
                ]
                sse = "".join(f"event: {e['type']}\ndata: {json.dumps(e)}\n\n" for e in events)
                return self._send(200, sse.encode(), "text/event-stream")
            return self._send(200, {
                "model": body["model"], "stop_reason": "end_turn",
                "content": [{"type": "text", "text": "Hello from hosted"}],
                "usage": {"input_tokens": 7, "output_tokens": 3}})
        if self.path == "/api/chat":
            if body["stream"]:
                lines = [{"message": {"content": "Hel"}, "done": False},
                         {"message": {"content": "lo"}, "done": False},
                         {"message": {"content": ""}, "done": True, "done_reason": "stop"}]
                return self._send(200, "\n".join(map(json.dumps, lines)).encode(), "application/x-ndjson")
            return self._send(200, {
                "model": body["model"], "done": True, "done_reason": "length",
                "message": {"role": "assistant", "content": "Hello from local"},
                "prompt_eval_count": 9, "eval_count": 4})
        if self.path.startswith("/v1beta/models/"):
            if ":streamGenerateContent" in self.path:
                chunks = [{"candidates": [{"content": {"parts": [{"text": t}]}}]} for t in ("Hel", "lo")]
                sse = "".join(f"data: {json.dumps(c)}\r\n\r\n" for c in chunks)
                return self._send(200, sse.encode(), "text/event-stream")
            return self._send(200, {
                "candidates": [{"content": {"role": "model", "parts": [{"text": "Hello from gemini"}]},
                                "finishReason": "STOP"}],
                "usageMetadata": {"promptTokenCount": 6, "candidatesTokenCount": 2},
                "modelVersion": self.path.split("/")[-1].split(":")[0]})
        self._send(404, {"error": "not found"})


class Tests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.srv = ThreadingHTTPServer(("127.0.0.1", 0), FakeHandler)
        cls.url = f"http://127.0.0.1:{cls.srv.server_address[1]}"
        threading.Thread(target=cls.srv.serve_forever, daemon=True).start()

    @classmethod
    def tearDownClass(cls):
        cls.srv.shutdown()

    def setUp(self):
        FakeHandler.fail_next = []
        self.hosted = AnthropicClient("claude-test", api_key="k", base_url=self.url)
        self.local = OllamaClient("llama-test", base_url=self.url)
        self.gemini = GeminiClient("gemini-test", api_key="g", base_url=self.url)

    def test_both_satisfy_protocol(self):
        for c in (self.hosted, self.gemini, self.local, FallbackClient(self.local)):
            self.assertIsInstance(c, LLMClient)

    def test_same_call_both_providers(self):
        msgs = [Message.system("sys-msg"), Message.user("hi")]
        h = self.hosted.chat(msgs, system="kwarg", max_tokens=50, temperature=0.2, stop=["X"])
        b = FakeHandler.last_body
        self.assertEqual(b["system"], "kwarg\n\nsys-msg")
        self.assertEqual(b["messages"], [{"role": "user", "content": "hi"}])
        self.assertEqual(b["stop_sequences"], ["X"])
        self.assertEqual(FakeHandler.last_headers["x-api-key"], "k")
        self.assertEqual((h.text, h.finish_reason, h.usage.input_tokens), ("Hello from hosted", "stop", 7))

        l = self.local.chat(msgs, system="kwarg", max_tokens=50, temperature=0.2, stop=["X"])
        b = FakeHandler.last_body
        self.assertEqual(b["messages"][0], {"role": "system", "content": "kwarg\n\nsys-msg"})
        self.assertEqual(b["options"], {"num_predict": 50, "temperature": 0.2, "stop": ["X"]})
        self.assertEqual((l.text, l.finish_reason, l.usage.output_tokens), ("Hello from local", "length", 4))

    def test_gemini(self):
        msgs = [Message.system("sys-msg"), Message.user("hi"), Message.assistant("yo"), Message.user("again")]
        r = self.gemini.chat(msgs, system="kwarg", max_tokens=50, temperature=0.2, stop=["X"])
        b = FakeHandler.last_body
        self.assertEqual(b["systemInstruction"], {"parts": [{"text": "kwarg\n\nsys-msg"}]})
        self.assertEqual([c["role"] for c in b["contents"]], ["user", "model", "user"])
        self.assertEqual(b["generationConfig"], {"maxOutputTokens": 50, "temperature": 0.2, "stopSequences": ["X"]})
        self.assertEqual(FakeHandler.last_headers["x-goog-api-key"], "g")
        self.assertEqual((r.text, r.model, r.finish_reason, r.usage.output_tokens),
                         ("Hello from gemini", "gemini-test", "stop", 2))
        self.assertEqual("".join(self.gemini.stream([Message.user("hi")])), "Hello")

    def test_streaming_both(self):
        self.assertEqual("".join(self.hosted.stream([Message.user("hi")])), "Hello")
        self.assertEqual("".join(self.local.stream([Message.user("hi")])), "Hello")

    def test_retries_then_succeeds(self):
        FakeHandler.fail_next = [529, 503]
        self.assertEqual(self.hosted.chat([Message.user("hi")]).text, "Hello from hosted")

    def test_non_retryable_raises(self):
        FakeHandler.fail_next = [400]
        with self.assertRaises(LLMError) as cm:
            self.hosted.chat([Message.user("hi")])
        self.assertEqual((cm.exception.status, cm.exception.retryable), (400, False))

    def test_fallback_local_down_uses_hosted(self):
        dead = OllamaClient("x", base_url="http://127.0.0.1:1", max_retries=0)
        r = FallbackClient(dead, self.hosted).chat([Message.user("hi")])
        self.assertEqual(r.provider, "anthropic")
        self.assertEqual("".join(FallbackClient(dead, self.local).stream([Message.user("hi")])), "Hello")

    def test_factory(self):
        c = create_client("ollama:llama3.2:3b", base_url=self.url)
        self.assertEqual((c.provider, c.model), ("ollama", "llama3.2:3b"))
        self.assertEqual(create_client("anthropic", api_key="k").model, "claude-sonnet-5-5")
        self.assertEqual(create_client("gemini", api_key="g").provider, "gemini")
        with self.assertRaises(ValueError):
            create_client("nope:model")


if __name__ == "__main__":
    unittest.main()

Writing test_llm_client.py


In [10]:
!python -m unittest -v test_llm_client

test_both_satisfy_protocol (test_llm_client.Tests.test_both_satisfy_protocol) ... ok
test_factory (test_llm_client.Tests.test_factory) ... ok
test_fallback_local_down_uses_hosted (test_llm_client.Tests.test_fallback_local_down_uses_hosted) ... ok
test_gemini (test_llm_client.Tests.test_gemini) ... ok
test_non_retryable_raises (test_llm_client.Tests.test_non_retryable_raises) ... ok
test_retries_then_succeeds (test_llm_client.Tests.test_retries_then_succeeds) ... ok
test_same_call_both_providers (test_llm_client.Tests.test_same_call_both_providers) ... ok
test_streaming_both (test_llm_client.Tests.test_streaming_both) ... ok

----------------------------------------------------------------------
Ran 8 tests in 2.253s

OK


## 3. Install and start Ollama inside Colab

In [11]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
import os, subprocess, time, urllib.request

# Start the Ollama server in the background (logs go to ollama.log)
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"),
                               stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})

for _ in range(60):  # wait up to ~30 s for it to come up
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1); break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("Ollama did not start — check ollama.log")
print("Ollama is running")

Ollama is running


Pull a model. `llama3.2` (3B, ~2 GB) is a good default for a T4; try `llama3.2:1b` on CPU, or `qwen2.5:7b` / `llama3.1:8b` if you want something larger.

In [13]:
MODEL = "llama3.2"
!ollama pull {MODEL}

## 4. Use the local model

In [14]:
import importlib
try:
    import llm_client
    importlib.reload(llm_client)
except ModuleNotFoundError:
    # If not yet written, running cell 1 will create it
    pass

from llm_client import create_client, Message

local = create_client(f"ollama:{MODEL}", base_url="http://127.0.0.1:11434")
resp = local.chat([Message.user("In one sentence, what is Google Colab?")], max_tokens=100)
print(resp.text)
print(resp.provider, resp.model, resp.finish_reason, resp.usage)

Google Colab is a free, cloud-based platform that allows users to create and share interactive computing documents, often used for data science, machine learning, and programming projects, using a Jupyter Notebook interface.
ollama llama3.2 stop Usage(input_tokens=35, output_tokens=42)


In [15]:
for chunk in local.stream([Message.user("Count from 1 to 10, comma separated.")], max_tokens=60):
    print(chunk, end="", flush=True)

1, 2, 3, 4, 5, 6, 7, 8, 9, 10

## 5. Use Gemini (hosted)

Get a free key at aistudio.google.com, then add it in Colab's **Secrets** panel (🔑 icon on the left) as `GEMINI_API_KEY` and turn on *Notebook access*.

In [16]:
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

gemini = create_client("gemini:gemini-3.5-flash-lite")
resp = gemini.chat([Message.user("In one sentence, what is Ai?")], max_tokens=100)
print(resp.text)
print(resp.provider, resp.model, resp.finish_reason, resp.usage)

Artificial intelligence (AI) is the simulation of human intelligence by computer systems, enabling them to learn, reason, perceive, and perform tasks that typically require human cognition.
gemini gemini-3.5-flash-lite stop Usage(input_tokens=9, output_tokens=33)


Colab VMs reset when the session ends, so steps 3–4 (install, start, pull) need re-running in each new session.